### Installation

In [ ]:
%%capture
import os

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = "YOUR_HF_ACCESS_TOKEN", # HF Token for gated models
)

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    modules_to_save=['embed_tokens', 'lm_head'],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

<a name="Data"></a>
### Data Prep
We now use the `Llama-3.1` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. But we convert it to HuggingFace's normal multiturn format `("role", "content")` instead of `("from", "value")`/ Llama-3 renders multi turn conversations like below:

```
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Hello!<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Hey there! How are you?<|eot_id|><|start_header_id|>user<|end_header_id|>

I'm great thanks!<|eot_id|>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3` and more.

In [ ]:
import pandas as pd

# Chọn uncomment 1 trong các file sau để chọn training set tương ứng cho từng thực nghiệm
df_train = pd.read_csv(r"/root/train.csv")                              # w/o augmentation
# df_train = pd.read_csv(r"/root/train_eda.csv")                          # EDA augmentation
# df_train = pd.read_csv(r"/root/train_aeda.csv")                         # AEDA augmentation
# df_train = pd.read_csv(r"/root/train_ease.csv")                         # EASE augmentation
# df_train = pd.read_csv(r"/root/train-llm-aug-gpt-5-2025-08-07.csv")     # LLM augmentation (gpt-5-2025-08-07)

df_val = pd.read_csv("/root/val.csv")
df_test = pd.read_csv("/root/test.csv")

# ===== UNDER_SAMPLING WITH REAL TRAINING DATA =====
# def process_training_set_with_balance_class_distribution(dataframe, target_column_name="priority"):
#     min_sample = dataframe[target_column_name].value_counts().min()
    
#     df_filtered = dataframe.groupby(target_column_name, group_keys=False).apply(
#         lambda x: x.sample(n=min_sample, random_state=42)
#     ).reset_index(drop=True)
    
#     return df_filtered

# df_train = process_training_set_with_balance_class_distribution(dataframe=df_train)

# ===== OVER_SAMPLING WITH REAL TRAINING DATA =====
# def process_training_set_with_oversampling(dataframe, target_column_name="priority"):
#     max_sample = dataframe[target_column_name].value_counts().max()
    
#     df_oversampled = dataframe.groupby(target_column_name, group_keys=False).apply(
#         lambda x: x.sample(n=max_sample, replace=True, random_state=42)
#     ).reset_index(drop=True)
    
#     return df_oversampled

# df_train = process_training_set_with_oversampling(dataframe=df_train)

In [ ]:
SYSTEM_PROMPT = "You are an expert software engineer with extensive experience in bug triaging and severity classification. You can accurately assess the impact and priority of software defects based on bug reports."
USER_PROMPT = """Bug Report:
{bug_content}

Based on the bug report above, classify it into one of the following severity levels:
- Blocker: Critical bugs that block development/testing progress or cause system crashes
- Critical: Major functionality broken, no workaround available, severe impact on users
- Major: Major functionality affected but workarounds exist, significant impact
- Minor: Minor functionality issues, cosmetic problems, or low impact bugs
- Trivial: Trivial issues, very minor cosmetic problems, or documentation typos

Provide your answer:"""

We now use `standardize_sharegpt` to convert ShareGPT style datasets into HuggingFace's generic format. This changes the dataset from looking like:
```
{"from": "system", "value": "You are an assistant"}
{"from": "human", "value": "What is 2+2?"}
{"from": "gpt", "value": "It's 4."}
```
to
```
{"role": "system", "content": "You are an assistant"}
{"role": "user", "content": "What is 2+2?"}
{"role": "assistant", "content": "It's 4."}
```

In [ ]:
def generate_train_instruction(system_prompt, user_prompt, bug_content, priority):
    instruction = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{user_prompt.format(bug_content=bug_content)}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{priority}<|eot_id|>"
    
    return instruction

def generate_test_instruction(system_prompt, user_prompt, bug_content):
    instruction = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{user_prompt.format(bug_content=bug_content)}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
    return instruction

In [ ]:
from tqdm import tqdm

# Generate instructions for training set
for index, values in tqdm(df_train.iterrows(), total=len(df_train), desc="Generating instruction prompt for training..."):
    BUG_CONTENT = df_train['text'][index]  # Sử dụng cột 'text'
    PRIORITY = df_train['priority'][index]  # Sử dụng cột 'priority'
    
    df_train.at[index, "instruction"] = generate_train_instruction(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=USER_PROMPT,
        bug_content=BUG_CONTENT,
        priority=PRIORITY
    )

# Generate instructions for validation set
for index, values in tqdm(df_val.iterrows(), total=len(df_val), desc="Generating instruction prompt for validation..."):
    BUG_CONTENT = df_val['text'][index]
    PRIORITY = df_val['priority'][index]
    
    df_val.at[index, "instruction"] = generate_train_instruction(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=USER_PROMPT,
        bug_content=BUG_CONTENT,
        priority=PRIORITY
    )

# Generate instructions for test set
for index, values in tqdm(df_test.iterrows(), total=len(df_test), desc="Generating instruction prompt for test..."):
    PRIORITY = df_test['text'][index]
    
    df_test.at[index, "instruction"] = generate_test_instruction(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=USER_PROMPT,
        bug_content=BUG_CONTENT
    )

In [ ]:
print(df_train.iloc[0]["instruction"])

In [ ]:
df_train["len_instruction"] = df_train["instruction"].apply(lambda x: len(x.split(" ")))
df_val["len_instruction"] = df_val["instruction"].apply(lambda x: len(x.split(" ")))
df_test["len_instruction"] = df_test["instruction"].apply(lambda x: len(x.split(" ")))

In [ ]:
df_val["len_instruction"].max()

In [ ]:
from datasets import Dataset, load_dataset

dataset_train = Dataset.from_pandas(df_train, preserve_index=False)
dataset_val = Dataset.from_pandas(df_val, preserve_index=False)
dataset_test = Dataset.from_pandas(df_test, preserve_index=False)

In [ ]:
def convert_to_text_format(dataset):
    def map_func(examples):
        return {"text": examples["instruction"]}
    
    return dataset.map(map_func, batched=True, remove_columns=dataset.column_names)

dataset_train_formatted = convert_to_text_format(dataset_train)
dataset_val_formatted = convert_to_text_format(dataset_val)
dataset_test_formatted = convert_to_text_format(dataset_test)

# Simple formatting function
def formatting_func(examples):
    return {"text": examples["text"]}

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_train_formatted,
    eval_dataset=dataset_val_formatted,
    dataset_text_field="text",
    max_seq_length=4096,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc=2,
    packing=False, # Can make training 5x faster for short sequences.
    formatting_func=formatting_func,
    args=TrainingArguments(
        per_device_train_batch_size=1, # 8
        gradient_accumulation_steps=1, # 8
        warmup_steps=100,
        num_train_epochs=1,
        learning_rate=2e-4, # base 2e-5
        fp16=True,
        bf16=False,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=10, 
        optim="paged_adamw_32bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        save_strategy="no",
    ),
)

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

We verify masking is actually done:

In [ ]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

We can see the System and Instruction prompts are successfully masked!

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!



We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [ ]:
FastLanguageModel.for_inference(model)

df_test["generated_answer"] = ""

for index, values in tqdm(df_test.iterrows(), total=len(df_test), desc="Generating answer for test set..."):
    instruction = values["instruction"]
    gold_answer = values["priority"]

    inputs = tokenizer(
        instruction,
        return_tensors='pt',
        truncation=True,
        max_length=32000
    ).to('cuda')

    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=20,
        # stopping_criteria=stopping_criteria,
        use_cache=True
    )

    generated_answer = tokenizer.batch_decode(outputs)[0].split("<|start_header_id|>assistant<|end_header_id|>\n\n")[1].replace("<|eot_id|>", "")

    df_test.at[index, "generated_answer"] = generated_answer
    
    print(f'==================== Generate output: ====================\n{generated_answer}')
    print(f"==================== Ground truth:====================\n{values['priority']}\n")

In [ ]:
df_test

In [ ]:
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# messages = [
#     {"role": "user", "content": "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,"},
# ]
# inputs = tokenizer.apply_chat_template(
#     messages,
#     tokenize = True,
#     add_generation_prompt = True, # Must add for generation
#     return_tensors = "pt",
# ).to("cuda")

# from transformers import TextStreamer
# text_streamer = TextStreamer(tokenizer, skip_prompt = True)
# _ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
#                    use_cache = True, temperature = 1.5, min_p = 0.1)

In [ ]:
df_test["generated_answer"].value_counts()

In [ ]:
df_test = df_test[["priority", "generated_answer"]]
# df_test["generated_answer"] = df_test["generated_answer"].apply(lambda x:x.replace("<think>\n\n</think>\n\n", "")).apply(lambda x:x.replace("<|im_end|>", ""))
df_test

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

In [ ]:
df_test["priority"] = le.fit_transform(df_test["priority"])
df_test["generated_answer"] = le.transform(df_test["generated_answer"])
df_test

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
print("Accuracy:", round(accuracy_score(df_test["priority"], df_test["generated_answer"])*100, 2))
print("Precision Macro:", round(precision_score(df_test["priority"], df_test["generated_answer"], average='macro')*100, 2))
print("Recall Macro:", round(recall_score(df_test["priority"], df_test["generated_answer"], average='macro')*100, 2))
print("F1 score Macro:", round(f1_score(df_test["priority"], df_test["generated_answer"], average='macro')*100, 2))

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
print("Accuracy:", round(accuracy_score(df_test["priority"], df_test["generated_answer"])*100, 2))
print("Precision Micro:", round(precision_score(df_test["priority"], df_test["generated_answer"], average='micro')*100, 2))
print("Recall Micro:", round(recall_score(df_test["priority"], df_test["generated_answer"], average='micro')*100, 2))
print("F1 score Micro:", round(f1_score(df_test["priority"], df_test["generated_answer"], average='micro')*100, 2))